In [ ]:
# =============================================
# Cell 1 — Import libraries
# =============================================

import os
import numpy as np
import pylidc as pl
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# =============================================
# Cell 2 — Set up paths and settings
# =============================================

# Where your raw CT scans 
DATA_PATH = r'C:\Users\User\Desktop\Lung Cancer Detection\data\raw\LIDC-IDRI'

# Where we will save the processed patches
SAVE_PATH = r'C:\Users\User\Desktop\Lung Cancer Detection\data\processed'

# Size of each patch we cut out around a nodule
PATCH_SIZE = 64

# HU value range for lung tissue
HU_MIN = -1000
HU_MAX = 400

# Create save folder if it does not exist
os.makedirs(os.path.join(SAVE_PATH, 'patches'), exist_ok=True)

print(f"Data path:     {DATA_PATH}")
print(f"Save path:     {SAVE_PATH}")
print(f"Patch size:    {PATCH_SIZE}x{PATCH_SIZE}x{PATCH_SIZE}")
print(f"HU range:      {HU_MIN} to {HU_MAX}")
print("\nSetup complete!")

In [ ]:
# =============================================
# Cell 3 — Load all scans
# =============================================

# Tell pylidc where the data is
#os.environ['LIDC_IDRI_PATH'] = DATA_PATH

# Load all scans
#scans = pl.query(pl.Scan).all()

#print(f"Total scans loaded: {len(scans)}")


# Load only 50 scans for now
# enough to build and test the CNN
scans = pl.query(pl.Scan).all()
scans = scans[:50]  # take first 50 only
print(f"Using {len(scans)} scans for development")

In [ ]:
# Cell 3: Compatibility Patch
import configparser
configparser.SafeConfigParser = configparser.ConfigParser
print("✅ Python 3.13 compatibility patch applied.")

In [ ]:
# Cell 4: Windows Configuration
from pathlib import Path

# Create the .pylidcrc file in your Windows Home directory
home = str(Path.home())
config_path = os.path.join(home, '.pylidcrc')

# Use the DATA_PATH you defined in Cell 2
config_content = f"""
[dicom]
path = {DATA_PATH}
warn = True
"""

with open(config_path, 'w') as f:
    f.write(config_content)

print(f"✅ Configuration saved to: {config_path}")

In [ ]:
# Cell 5: Helper Functions
def clip_and_normalise(patch):
    # Standardize lung density (HU units)
    patch = np.clip(patch, HU_MIN, HU_MAX)
    return (patch - HU_MIN) / (HU_MAX - HU_MIN)

def extract_patch(volume, center_x, center_y, center_z, size=64):
    half = size // 2
    # Define boundaries
    x_s, x_e = int(max(0, center_x-half)), int(min(volume.shape[0], center_x+half))
    y_s, y_e = int(max(0, center_y-half)), int(min(volume.shape[1], center_y+half))
    z_s, z_e = int(max(0, center_z-half)), int(min(volume.shape[2], center_z+half))
    
    patch = volume[x_s:x_e, y_s:y_e, z_s:z_e]
    
    # Pad with zeros if the patch is near the edge of the lung
    padded = np.zeros((size, size, size))
    padded[:patch.shape[0], :patch.shape[1], :patch.shape[2]] = patch
    return padded

print("✅ Helper functions ready.")

In [ ]:
# Cell 6: The Big Extraction
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

# NumPy 2.x compatibility for older pylidc internals
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "bool"):
    np.bool = bool

# Use the scans prepared earlier (Cell 3).
# Fallback to a small subset if scans is missing.
if "scans" in globals() and len(scans) > 0:
    scans_to_process = scans
else:
    scans_to_process = pl.query(pl.Scan).all()[:50]

patches_list, labels_list = [], []
count = 0
processed_ok = 0
error_count = 0
error_examples = {}

print(f"Starting extraction for {len(scans_to_process)} scans...")

for scan in scans_to_process:
    # Check if this specific patient folder exists on your Desktop
    if not os.path.exists(os.path.join(DATA_PATH, scan.patient_id)):
        continue

    try:
        vol = scan.to_volume()  # Load 3D pixels
        nodules = scan.cluster_annotations()  # Find radiologist marks

        for nodule in nodules:
            # Malignancy 1-2 = Benign (0), 3-5 = Malignant (1)
            avg_malig = np.mean([ann.malignancy for ann in nodule])
            label = 1 if avg_malig >= 3 else 0

            # Get coordinates and crop
            cx, cy, cz = nodule[0].centroid
            patch = extract_patch(vol, cx, cy, cz, size=PATCH_SIZE)
            patch = clip_and_normalise(patch)

            # Save locally to data/processed/patches
            fname = f"patch_{count:04d}.npy"
            save_file_path = os.path.join(SAVE_PATH, "patches", fname)
            np.save(save_file_path, patch)

            patches_list.append(fname)
            labels_list.append(label)
            count += 1

        processed_ok += 1
        print(f"Processed {scan.patient_id}")

    except Exception as e:
        error_count += 1
        msg = str(e).splitlines()[0]
        error_examples[msg] = error_examples.get(msg, 0) + 1

# Save the final label file even if partial extraction succeeded
df = pd.DataFrame({"patch_file": patches_list, "label": labels_list})
df.to_csv(os.path.join(SAVE_PATH, "labels.csv"), index=False)

print("\nExtraction summary")
print(f"Scans succeeded: {processed_ok}")
print(f"Scans failed:    {error_count}")
print(f"Patches saved:   {count}")
print(f"Saved to:        {SAVE_PATH}")

if error_examples:
    print("\nTop error types:")
    for msg, n in sorted(error_examples.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"- {n}x {msg}")

In [ ]:
# =============================================
# Cell 7 — Verify saved patches
# =============================================

# Check the patches folder
patch_folder = os.path.join(SAVE_PATH, 'patches')
saved_files = os.listdir(patch_folder)
print(f"Files in patches folder: {len(saved_files)}")

# Load one patch and check its shape
sample_patch = np.load(os.path.join(patch_folder, saved_files[0]))
print(f"Sample patch shape: {sample_patch.shape}")
print(f"Min value: {sample_patch.min():.3f}")
print(f"Max value: {sample_patch.max():.3f}")
print(f"\nShape should be: (64, 64, 64)")
print(f"Min should be:   0.0")
print(f"Max should be:   1.0")

In [ ]:
# Cell 8 — Visualize a processed patch
import matplotlib.pyplot as plt

# Load the labels CSV we created
df = pd.read_csv(os.path.join(SAVE_PATH, 'labels.csv'))

# Find one cancer (1) and one benign (0) patch
cancer_file = df[df['label'] == 1]['patch_file'].iloc[0]
benign_file = df[df['label'] == 0]['patch_file'].iloc[0]

# Load the actual .npy files
cancer_patch = np.load(os.path.join(SAVE_PATH, 'patches', cancer_file))
benign_patch = np.load(os.path.join(SAVE_PATH, 'patches', benign_file))

# Plotting the middle slice (index 32)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(cancer_patch[:, :, 32], cmap='gray')
axes[0].set_title(f'Cancer Patch\n{cancer_file}')
axes[0].axis('off')

axes[1].imshow(benign_patch[:, :, 32], cmap='gray')
axes[1].set_title(f'Benign Patch\n{benign_file}')
axes[1].axis('off')

plt.suptitle('Final Processed Patches — Ready for CNN', fontsize=14)
plt.show()

print("\n✅ Preprocessing Stage Complete!")

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Define your paths (Make sure these match your project folder)
BASE_PATH = r'C:\Users\User\Desktop\Lung Cancer Detection'
CSV_PATH = os.path.join(BASE_PATH, 'data', 'processed', 'labels.csv')
PATCH_PATH = os.path.join(BASE_PATH, 'data', 'processed', 'patches')

# Load labels
df = pd.read_csv(CSV_PATH)

# Split into train and validation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"Total patches: {len(df)}")
print(f"Training on: {len(train_df)}")
print(f"Validating on: {len(val_df)}")

In [ ]:
# Cell 10 — TensorFlow-ready Data Generator
import os
import numpy as np
import pandas as pd
import gc

def data_generator(dataframe, batch_size=4, patch_size=64):
    patches_processed = 0
    while True:
        # Shuffle once per epoch
        df_shuffled = dataframe.sample(frac=1).reset_index(drop=True)
        yielded_in_epoch = False

        for i in range(0, len(df_shuffled), batch_size):
            batch_df = df_shuffled.iloc[i:i + batch_size]
            X, y = [], []

            for _, row in batch_df.iterrows():
                path = os.path.join(PATCH_PATH, row["patch_file"])
                if not os.path.exists(path):
                    continue

                try:
                    # Use mmap_mode='r' to prevent loading the entire array into RAM at once
                    patch = np.load(path, mmap_mode='r')
                except Exception:
                    continue

                # Ensure every patch is exactly 64x64x64 before adding channel dim
                if patch.shape != (patch_size, patch_size, patch_size):
                    continue

                # Copying into memory as float32 only for the valid chunks requested
                X.append(np.array(patch, dtype=np.float32))
                y.append(int(row["label"]))
                patches_processed += 1

            if len(X) == 0:
                continue

            # X shape: (batch, 64, 64, 64) -> (batch, 64, 64, 64, 1)
            X_batch = np.expand_dims(np.stack(X, axis=0), axis=-1)
            y_batch = np.array(y, dtype=np.int32)

            yielded_in_epoch = True
            yield X_batch, y_batch
            
            # Explicit cleanup and garbage collection
            del X_batch, y_batch, X, y
            if patches_processed >= 10:
                gc.collect()
                patches_processed = 0

        if not yielded_in_epoch:
            raise RuntimeError(
                "No valid patches found. Check PATCH_PATH, labels.csv, and patch shape (64,64,64)."
            )

train_gen = data_generator(train_df, batch_size=4)
val_gen = data_generator(val_df, batch_size=4)
print("Generators ready with memory optimization!")

In [ ]:
# Cell 11 — Generator sanity check
sample_x, sample_y = next(train_gen)
print("Generator check passed.")
print("X batch shape:", sample_x.shape)
print("y batch shape:", sample_y.shape)

In [ ]:
# Cell 12 — 3D CNN Model (TensorFlow Keras)
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models

    model = models.Sequential([
        layers.Input(shape=(64, 64, 64, 1)),
        layers.Conv3D(32, kernel_size=3, activation="relu", padding="same"),
        layers.MaxPooling3D(pool_size=2),
        layers.Conv3D(64, kernel_size=3, activation="relu", padding="same"),
        layers.MaxPooling3D(pool_size=2),
        layers.Conv3D(128, kernel_size=3, activation="relu", padding="same"),
        layers.MaxPooling3D(pool_size=2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()

except ModuleNotFoundError:
    print("TensorFlow is not installed in this environment.")
    print("Cell 10 and Cell 11 can still run without TensorFlow.")
    print("Install later with: pip install tensorflow")